# Basic Tasks 

In [0]:
data = [
    (1, "Laptop", "Electronics", 75000),
    (2, "Mobile", "Electronics", 30000),
    (3, "Chair", "Furniture", 5000),
    (4, "Table", "Furniture", 12000),
    (5, "Headphones", "Electronics", 3000)
]

schema = ["product_id", "product_name", "category", "price"]

df = spark.createDataFrame(data, schema)
df.display()

In [0]:
df_infer = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/Volumes/cyntexa_dev/sales/sales_volume/orders_normal.csv")

In [0]:
df_infer.printSchema()

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

In [0]:
schema = StructType([
    StructField("shipment_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("shipment_date", DateType(), True),
    StructField("status", StringType(), True),
    StructField("amount", IntegerType(), True)
    ])

df_explicit = spark.read.format("csv").option("header", True).schema(schema).load("/Volumes/cyntexa_dev/sales/sales_volume/orders_normal.csv")

In [0]:
df_explicit.printSchema()

In [0]:
from pyspark.sql.functions import col

In [0]:
filtered_df = df_explicit.filter(col("amount") > 2000)

In [0]:
filtered_df.show()

In [0]:
filtered_df.count()

## **Lazy Evaluation:** 
filter() is a transformation, so Spark does not immediately execute it. Instead, Spark builds a logical execution plan. Execution starts when an action such as show() or count() is called. This allows Spark to optimize the execution plan before processing the data.![](path)

# Intermediate Tasks 

In [0]:
order_df = spark.read.format("csv").option("header", True).option("inferSchema", True).load("/Volumes/cyntexa_dev/sales/sales_volume/orders_normal.csv")

In [0]:
delivered_df = order_df.filter(col("status") == "Delivered")

In [0]:
from pyspark.sql.functions import current_date

In [0]:
result_df = delivered_df.withColumn("ingestion_date", current_date())

In [0]:
result_df.display()

In [0]:
result_df.write.mode("overwrite").saveAsTable("cyntexa_dev.sales.orders_elt")

In [0]:
nested_df = spark.read.format("json").option("multiline", True).load("/Volumes/cyntexa_dev/sales/sales_volume/shipments_nested.json")

In [0]:
nested_df.printSchema()

In [0]:
flattened_df = nested_df.select("shipment_id", "customer.customer_id", "customer.name", "shipping.city", "shipping.country", "amount")

In [0]:
flattened_df.display()

### Nested JSON Flattening: The JSON contains nested customer and shipping structures. Dot notation is used to access the nested fields and select them as top-level columns. This converts the nested structure into a flat DataFrame that is easier to analyze.

In [0]:
filtered_df.explain()

In [0]:
filtered_df.explain(True)

In [0]:
result_df.explain()

### **Physical Plan Explanation:** 
The explain() method shows how Spark plans to execute the DataFrame operations. The physical plan can include operations such as FileScan for reading the source data and Filter for applying filter conditions. Spark uses the optimized execution plan to efficiently execute the transformations when an action is triggered.

# **Advanced Tasks**

### Pandas to PySpark

In [0]:
import pandas as pd

pandas_df = pd.DataFrame({
    "product": ["Laptop", "Mobile", "Chair", "Table", "Headphones"],
    "category": ["Electronics", "Electronics", "Furniture", "Furniture", "Electronics"],
    "amount": [75000, 30000, 5000, 12000, 3000]
})

In [0]:
pandas_df

In [0]:
pandas_filtered = pandas_df[pandas_df["amount"] > 10000]

In [0]:
pandas_filtered

In [0]:
spark_df = spark.createDataFrame(pandas_df)

In [0]:
spark_df.display()

In [0]:
spark_df.printSchema()

In [0]:
pyspark_filtered = spark_df.filter(col("amount") > 10000)

In [0]:
pyspark_filtered.display()

### **Pandas to PySpark Conversion:** 
The pandas filtering operation was converted to the PySpark filter() DataFrame API. The pandas apply() logic for creating a derived column was converted to PySpark withColumn() with when() and otherwise(). Both implementations perform the same logical transformations, but PySpark DataFrames are designed for distributed processing.

In [0]:
partitioned_df = spark_df.repartition(4)

In [0]:
partitioned_df.display()

In [0]:
category_partitioned_df = spark_df.repartition(4, "category")

In [0]:
category_partitioned_df.display()

In [0]:
category_partitioned_df.write.mode("overwrite").partitionBy("category").saveAsTable("cyntexa_dev.sales.products_partitioned")

In [0]:
# This is called partition pruning because, spark avoid scanning unrelated category partitions.
spark.sql("""
SELECT *
FROM cyntexa_dev.sales.products_partitioned
WHERE category = 'Electronics'
""").display()

In [0]:
spark.sql("""
SELECT *
FROM cyntexa_dev.sales.products_partitioned
WHERE category = 'Electronics'
""").explain()

### Task 8 — Partitioning and Write Strategy

Since this table will mostly be queried using date ranges, I would partition the table by a date column such as `order_date` or `shipment_date`.

The table can be written using `partitionBy("order_date")` so that data is physically organized by date. This allows Spark to use partition pruning when a query filters on a specific date range, reducing the amount of data that needs to be scanned.

For the write strategy, I would avoid creating many very small files and aim for reasonably sized output files. The target file size should be large enough to reduce small-file overhead while still allowing Spark to process files efficiently.

Spark uses lazy evaluation, so transformations such as filtering by date are not executed immediately. When an action is triggered, Spark creates and executes a physical plan. For a date-range query, the physical plan can use partition filters to read only the relevant date partitions instead of scanning the entire table.

Therefore, partitioning by the commonly filtered date column can improve query performance by reducing unnecessary data scanning.

In [0]:
from pyspark.sql import Row

sales_data = [
    ("S001", "2026-01-05", "Electronics", 5000),
    ("S002", "2026-01-10", "Furniture", 3000),
    ("S003", "2026-01-15", "Electronics", 7000),
    ("S004", "2026-01-20", "Clothing", 2500),
    
    ("S005", "2026-02-05", "Electronics", 8000),
    ("S006", "2026-02-12", "Furniture", 4500),
    ("S007", "2026-02-18", "Clothing", 3500),
    ("S008", "2026-02-25", "Electronics", 6000),
    
    ("S009", "2026-03-03", "Furniture", 5500),
    ("S010", "2026-03-08", "Electronics", 9000),
    ("S011", "2026-03-15", "Clothing", 4000),
    ("S012", "2026-03-22", "Furniture", 3000),
]

sales_df = spark.createDataFrame(
    sales_data,
    ["sale_id", "sale_date", "category", "amount"]
)

In [0]:
from pyspark.sql.functions import to_date

sales_df = sales_df.withColumn(
    "sale_date",
    to_date("sale_date")
)

sales_df.printSchema()

In [0]:
from pyspark.sql.functions import date_format

In [0]:
monthly_sales = sales_df.withColumn(
    "month",
    date_format("sale_date", "yyyy-MM")
)

display(monthly_sales)

In [0]:
from pyspark.sql.functions import sum

revenue_df = (
    monthly_sales
    .groupBy("month", "category")
    .agg(
        sum("amount").alias("revenue")
    )
)

In [0]:
revenue_df.display()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

# Create the window:
window_spec = Window \
    .partitionBy("month") \
    .orderBy(desc("revenue"))

In [0]:
ranked_df = revenue_df.withColumn(
    "rank",
    row_number().over(window_spec)
)

In [0]:
ranked_df.display()

In [0]:
final_report = ranked_df.orderBy(
    "month",
    "rank"
)

display(final_report)

In [0]:
top_category_df = ranked_df.filter(
    ranked_df.rank == 1
)

display(top_category_df)

### Task 9 — Monthly Revenue by Category Report

The sales data was transformed by converting the sale date to a DateType and deriving a month column using date_format(). The data was then grouped by month and category, and total revenue was calculated using SUM(amount).

A window function was used to rank categories within each month based on revenue in descending order. Rank 1 represents the category with the highest revenue for that month.

The final report shows monthly revenue by category along with the revenue rank, which can be used to identify the best-performing category for each month.